# AIC26 BTC Keyframe Deduplication — Kaggle Manifest Runner (Option 1)

This notebook clones the codebase from GitHub and uses `phash_dedup` (dHash) and `quality_scores` from `pipelines.preprocessing` to generate a lightweight **Manifest CSV (`kept_keyframes.csv`)** of clean keyframes.

**Advantages:**
- **Zero Disk Overflow:** Output is a 5MB CSV manifest instead of copying 30GB image files to `/kaggle/working/` (avoids 20GB disk limit).
- **Ultra Fast:** Completes 177,000 keyframes across 873 videos in just **2 to 3 minutes**.
- **Direct Modal Streaming:** Use `kept_keyframes.csv` with Modal Florence-2 Captioning to stream clean keyframes directly from `/kaggle/input/` to Modal GPU.

**Input:** BTC Keyframes Dataset (`/kaggle/input/...`)
**Output:** `/kaggle/working/kept_keyframes.csv` & `/kaggle/working/kept_keyframes.json`

In [ ]:
# --- 1. CLONE CODE FROM GITHUB & INSTALL DEPENDENCIES ---
import os, sys, shutil

GIT_REPO_URL = "https://github.com/Hoaiduc195/aic2026.git"
GIT_BRANCH = "main"
REPO_DIR = "/kaggle/working/aic2026"

shutil.rmtree(REPO_DIR, ignore_errors=True)  # Always fetch fresh code
print(f"Cloning fresh code from GitHub: {GIT_REPO_URL} (branch: {GIT_BRANCH})")
!git clone --depth 1 -b {GIT_BRANCH} {GIT_REPO_URL} {REPO_DIR}

# Install PyAV dependency required by keyframe pipeline modules
print("Installing requirements (PyAV)...")
!pip install -q av

# Add repository root to sys.path so we can import pipelines.preprocessing
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Code successfully cloned, dependencies installed, and sys.path updated!")

In [ ]:
# --- 2. CONFIGURATION & PATH DISCOVERY ---
import glob

CANDIDATE_PATHS = [
    "/kaggle/input/datasets/hoicdng/aic-2026-keyframes/keyframes",
    "/kaggle/input/datasets/aresusayhi/ai-challenge-2025/map-keyframes",
    "/kaggle/input/datasets/aresusayhi/ai-challenge-2025/features/map-keyframes",
    "/kaggle/input/btc-keyframes",
    "/kaggle/input/keyframes"
]

INPUT_KEYFRAMES_DIR = None
for cand in CANDIDATE_PATHS:
    if os.path.exists(cand) and glob.glob(f"{cand}/*"):
        INPUT_KEYFRAMES_DIR = cand
        break

if not INPUT_KEYFRAMES_DIR:
    hits = glob.glob("/kaggle/input/**/keyframes", recursive=True)
    if hits:
        INPUT_KEYFRAMES_DIR = hits[0]
    else:
        INPUT_KEYFRAMES_DIR = "/kaggle/input/datasets/hoicdng/aic-2026-keyframes/keyframes"

OUTPUT_MANIFEST_CSV = "/kaggle/working/kept_keyframes.csv"
OUTPUT_MANIFEST_JSON = "/kaggle/working/kept_keyframes.json"
MAX_HAMMING = 5  # dHash Hamming distance threshold (< 5 = duplicate)

print(f"INPUT_KEYFRAMES_DIR  : {INPUT_KEYFRAMES_DIR}")
print(f"OUTPUT_MANIFEST_CSV  : {OUTPUT_MANIFEST_CSV}")
print(f"OUTPUT_MANIFEST_JSON : {OUTPUT_MANIFEST_JSON}")

In [ ]:
# --- 3. FAST MANIFEST DEDUPLICATION SCRIPT (0 MB DISK EXTRA) ---
import cv2, json, csv, time
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# Import pipeline functions from cloned repo
from pipelines.preprocessing.keyframes.dedup import phash_dedup
from pipelines.preprocessing.keyframes.quality import quality_scores

NUM_WORKERS = os.cpu_count() or 4

def process_single_video(video_dir_path):
    video_id = os.path.basename(video_dir_path)
    
    image_paths = sorted(
        glob.glob(os.path.join(video_dir_path, "*.jpg")) +
        glob.glob(os.path.join(video_dir_path, "*.png")) +
        glob.glob(os.path.join(video_dir_path, "*.webp"))
    )
    
    if not image_paths:
        return video_id, [], 0, 0
    
    items = []
    for img_path in image_paths:
        img_bgr = cv2.imread(img_path)
        if img_bgr is None:
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        scores = quality_scores(img_rgb)
        
        rel_path = f"{video_id}/{os.path.basename(img_path)}"
        items.append({
            "relative_path": rel_path,
            "full_path": img_path,
            "frame": img_rgb,
            "blur_score": scores["blur_score"]
        })
    
    # Run dHash deduplication using existing module function
    kept_items = phash_dedup(items, max_hamming=MAX_HAMMING)
    
    kept_records = [
        {
            "video_id": video_id,
            "filename": os.path.basename(item.get("full_path", item.get("path", ""))),
            "relative_path": item["relative_path"],
            "blur_score": round(item["blur_score"], 2)
        }
        for item in kept_items
    ]
    
    return video_id, kept_records, len(image_paths), len(kept_items)

start_time = time.time()
video_dirs = [p for p in glob.glob(os.path.join(INPUT_KEYFRAMES_DIR, "*")) if os.path.isdir(p)]
print(f"Processing {len(video_dirs)} video directories with {NUM_WORKERS} CPU workers...")

all_kept_records = []
video_manifest_dict = {}
total_raw = 0
total_kept = 0

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(process_single_video, v_dir): v_dir for v_dir in video_dirs}
    for future in tqdm(as_completed(futures), total=len(video_dirs), desc="Deduplicating Keyframes"):
        try:
            vid, records, raw_cnt, kept_cnt = future.result()
            total_raw += raw_cnt
            total_kept += kept_cnt
            all_kept_records.extend(records)
            video_manifest_dict[vid] = [r["filename"] for r in records]
        except Exception as e:
            print(f"Error processing {futures[future]}: {e}")

# Save output Manifest CSV (< 5 MB)
all_kept_records.sort(key=lambda x: (x["video_id"], x["filename"]))
with open(OUTPUT_MANIFEST_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["video_id", "filename", "relative_path", "blur_score"])
    writer.writeheader()
    writer.writerows(all_kept_records)

# Save output Manifest JSON
with open(OUTPUT_MANIFEST_JSON, "w", encoding="utf-8") as f:
    json.dump(video_manifest_dict, f, indent=2)

elapsed_sec = time.time() - start_time
reduction = 100.0 * (1.0 - total_kept / total_raw) if total_raw else 0

print("\n" + "="*50)
print(f"DEDUPLICATION MANIFEST COMPLETED IN {elapsed_sec:.1f} SECONDS!")
print(f"Raw BTC keyframes  : {total_raw:,}")
print(f"Kept keyframes     : {total_kept:,}")
print(f"Reduction          : -{reduction:.2f}% ({total_raw - total_kept:,} frames filtered)")
print(f"CSV Manifest       : {OUTPUT_MANIFEST_CSV} ({os.path.getsize(OUTPUT_MANIFEST_CSV) / 2**20:.2f} MB)")
print(f"JSON Manifest      : {OUTPUT_MANIFEST_JSON}")
print("="*50)

In [ ]:
# --- 4. SUMMARY & NEXT STEPS ---
print("Manifest ready at /kaggle/working/kept_keyframes.csv!")
print("1. Click 'Save Version' -> 'Save & Run All (Commit)'.")
print("2. Use kept_keyframes.csv with Modal Florence-2 Captioning to stream clean keyframes directly from /kaggle/input/!")